# Phase 3 — experiments (manual, fail-closed)
Run one cell at a time. Do not run all. No full model is downloaded before the independent approval gate. Runtime outputs and resumable checkpoints live under `Experiments/`.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('pip') is None:
    subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'], check=True)
%pip install -q -r requirements-lightning.txt
%pip install -q -e . --no-deps

In [ ]:
import importlib.metadata as md
import torch
assert md.version('torch') == '2.7.1', f"Select a Lightning image with torch 2.7.1; found {md.version('torch')}"
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

In [ ]:
from src.experiments.phase3_runner import prepare_phase3
dry_report = prepare_phase3(mode='dry_run', download_official_snapshot=False)
dry_report

In [ ]:
# Small official files only; no model and no GPU inference.
snapshot_report = prepare_phase3(mode='dry_run', download_official_snapshot=True)
snapshot_report['outputs']

## STOP — independent pre-run audit
Send the exact repository commit for independent review now. Continue only after an explicit PASS. This is the boundary before any credit-bearing model execution.

In [ ]:
audit_code = input('Type INDEPENDENT-AUDIT-PASS only after the audit passed: ').strip()
assert audit_code == 'INDEPENDENT-AUDIT-PASS', 'STOP: independent approval is required before GPU inference.'

In [ ]:
from src.experiments.phase3_runner import run_model_cell, run_precision_gate
smoke = run_model_cell('mamba_130m', mode='smoke', conditions=['clean', 'hispa_b'])
smoke

In [ ]:
precision = run_precision_gate(mode='smoke', model_key='mamba_130m')
precision

In [ ]:
FULL_CONFIRMATION = 'I_ACCEPT_PHASE3_H100_COST'
full_report = prepare_phase3(mode='full', confirmation=FULL_CONFIRMATION, download_official_snapshot=True)
full_report['preflight']

## Full run — one model per cell
Each cell resumes from its last atomic checkpoint and unloads the model before returning. Stop if the remaining budget is unexpectedly low.

In [ ]:
run_model_cell('mamba_130m', mode='full', confirmation=FULL_CONFIRMATION)

In [ ]:
run_model_cell('mamba_790m', mode='full', confirmation=FULL_CONFIRMATION)

In [ ]:
run_model_cell('mamba_1_4b', mode='full', confirmation=FULL_CONFIRMATION)

In [ ]:
run_model_cell('mamba_2_8b', mode='full', confirmation=FULL_CONFIRMATION)

In [ ]:
run_model_cell('pythia_1_4b', mode='full', confirmation=FULL_CONFIRMATION)

In [ ]:
run_model_cell('pythia_2_8b', mode='full', confirmation=FULL_CONFIRMATION)

In [ ]:
run_model_cell('gpt2_xl', mode='full', confirmation=FULL_CONFIRMATION)

In [ ]:
run_model_cell('zamba2_2_7b', mode='full', confirmation=FULL_CONFIRMATION)

In [ ]:
from src.experiments.phase3_runner import export_experiment_bundle
experiments_zip = export_experiment_bundle()
print(experiments_zip)